# Practico Parcial 1 - Regla MAP y simulacion Monte Carlo

Este notebook desarrolla el problema de prueba de hipotesis binaria aplicado a un sistema optico con conteo de fotones.

La idea es dejar el desarrollo matematico escrito paso a paso, como se haria en una hoja durante el parcial, y despues verificar el resultado con una simulacion Monte Carlo.

Datos del enunciado:

$$\lambda_0 = 1, \qquad \lambda_1 = 3$$

$$P_H(0)=0.7, \qquad P_H(1)=0.3$$

Modelo:

$$Y|H=0 \sim \text{Poisson}(\lambda_0)$$

$$Y|H=1 \sim \text{Poisson}(\lambda_1)$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import exp, factorial, log, ceil

plt.style.use("seaborn-v0_8-whitegrid")

rng = np.random.default_rng(20260507)

lambda0 = 1
lambda1 = 3
p0 = 0.7
p1 = 0.3
nb_samples = 1_000_000

## 1. Desarrollo matematico para obtener el nivel de decision MAP

Queremos decidir entre dos hipotesis:

$$H=0 \qquad \text{o} \qquad H=1$$

A la salida del canal se observa un valor discreto $Y=y$, que representa la cantidad de fotones detectados.

La regla MAP elige la hipotesis con mayor probabilidad a posteriori:

$$\hat H(y)=\arg\max_{i \in \{0,1\}} P_{H|Y}(i|y)$$

Para el caso binario, decidir $H=1$ significa que la probabilidad posterior de $H=1$ es mayor o igual que la de $H=0$:

$$\hat H = 1 \quad \text{si} \quad P_{H|Y}(1|y) \ge P_{H|Y}(0|y)$$

Usamos Bayes en cada miembro:

$$P_{H|Y}(1|y)=\frac{P_H(1)P_{Y|H}(y|1)}{P_Y(y)}$$

$$P_{H|Y}(0|y)=\frac{P_H(0)P_{Y|H}(y|0)}{P_Y(y)}$$

Reemplazando en la desigualdad:

$$\frac{P_H(1)P_{Y|H}(y|1)}{P_Y(y)} \ge \frac{P_H(0)P_{Y|H}(y|0)}{P_Y(y)}$$

Como $P_Y(y)$ es el mismo denominador en ambos lados y ademas es positivo, se cancela:

$$P_H(1)P_{Y|H}(y|1) \ge P_H(0)P_{Y|H}(y|0)$$

Ahora dividimos ambos lados por $P_H(0)P_{Y|H}(y|0)$, que tambien es positivo:

$$\frac{P_H(1)P_{Y|H}(y|1)}{P_H(0)P_{Y|H}(y|0)} \ge 1$$

Separamos el cociente en dos factores:

$$\frac{P_H(1)}{P_H(0)} \cdot \frac{P_{Y|H}(y|1)}{P_{Y|H}(y|0)} \ge 1$$

Esta es la forma de la regla MAP usando razon de verosimilitudes.


## 2. Sustitucion de las distribuciones de Poisson

La probabilidad de observar $Y=y$ cuando la intensidad es $\lambda$ esta dada por la pmf de Poisson:

$$P(Y=y)=\frac{\lambda^y}{y!}e^{-\lambda}, \qquad y=0,1,2,\dots$$

Entonces, para cada hipotesis:

$$P_{Y|H}(y|0)=\frac{\lambda_0^y}{y!}e^{-\lambda_0}$$

$$P_{Y|H}(y|1)=\frac{\lambda_1^y}{y!}e^{-\lambda_1}$$

Volvemos a la regla MAP:

$$\frac{P_H(1)}{P_H(0)} \cdot \frac{P_{Y|H}(y|1)}{P_{Y|H}(y|0)} \ge 1$$

Reemplazamos las pmf:

$$\frac{P_H(1)}{P_H(0)} \cdot \frac{\frac{\lambda_1^y}{y!}e^{-\lambda_1}}{\frac{\lambda_0^y}{y!}e^{-\lambda_0}} \ge 1$$

Para simplificar el cociente de fracciones, multiplicamos por la inversa del denominador:

$$\frac{P_H(1)}{P_H(0)} \cdot \frac{\lambda_1^y}{y!}e^{-\lambda_1} \cdot \frac{y!}{\lambda_0^y e^{-\lambda_0}} \ge 1$$

Cancelamos $y!$:

$$\frac{P_H(1)}{P_H(0)} \cdot \frac{\lambda_1^y e^{-\lambda_1}}{\lambda_0^y e^{-\lambda_0}} \ge 1$$

Separamos potencias y exponenciales:

$$\frac{P_H(1)}{P_H(0)} \cdot \frac{\lambda_1^y}{\lambda_0^y} \cdot \frac{e^{-\lambda_1}}{e^{-\lambda_0}} \ge 1$$

Usamos que $\frac{a^y}{b^y}=\left(\frac{a}{b}\right)^y$:

$$\frac{P_H(1)}{P_H(0)} \cdot \left(\frac{\lambda_1}{\lambda_0}\right)^y \cdot \frac{e^{-\lambda_1}}{e^{-\lambda_0}} \ge 1$$

Usamos que $\frac{e^a}{e^b}=e^{a-b}$:

$$\frac{e^{-\lambda_1}}{e^{-\lambda_0}}=e^{-\lambda_1-(-\lambda_0)}=e^{-\lambda_1+\lambda_0}=e^{-(\lambda_1-\lambda_0)}$$

Entonces:

$$\frac{P_H(1)}{P_H(0)} \cdot \left(\frac{\lambda_1}{\lambda_0}\right)^y \cdot e^{-(\lambda_1-\lambda_0)} \ge 1$$


## 3. Despeje del umbral, sin saltear pasos

Partimos de:

$$\frac{P_H(1)}{P_H(0)} \cdot \left(\frac{\lambda_1}{\lambda_0}\right)^y \cdot e^{-(\lambda_1-\lambda_0)} \ge 1$$

Aplicamos logaritmo natural a ambos lados. Como el logaritmo natural es creciente, no cambia el sentido de la desigualdad:

$$\ln\left[\frac{P_H(1)}{P_H(0)} \cdot \left(\frac{\lambda_1}{\lambda_0}\right)^y \cdot e^{-(\lambda_1-\lambda_0)}\right] \ge \ln(1)$$

Como $\ln(1)=0$:

$$\ln\left[\frac{P_H(1)}{P_H(0)} \cdot \left(\frac{\lambda_1}{\lambda_0}\right)^y \cdot e^{-(\lambda_1-\lambda_0)}\right] \ge 0$$

Usamos que el logaritmo de un producto es la suma de los logaritmos:

$$\ln\left(\frac{P_H(1)}{P_H(0)}\right) + \ln\left[\left(\frac{\lambda_1}{\lambda_0}\right)^y\right] + \ln\left(e^{-(\lambda_1-\lambda_0)}\right) \ge 0$$

Usamos que $\ln(a^y)=y\ln(a)$:

$$\ln\left(\frac{P_H(1)}{P_H(0)}\right) + y\ln\left(\frac{\lambda_1}{\lambda_0}\right) + \ln\left(e^{-(\lambda_1-\lambda_0)}\right) \ge 0$$

Usamos que $\ln(e^x)=x$:

$$\ln\left(\frac{P_H(1)}{P_H(0)}\right) + y\ln\left(\frac{\lambda_1}{\lambda_0}\right) - (\lambda_1-\lambda_0) \ge 0$$

Ahora queremos despejar $y$. Primero pasamos los terminos que no tienen $y$ al otro lado:

$$y\ln\left(\frac{\lambda_1}{\lambda_0}\right) \ge (\lambda_1-\lambda_0) - \ln\left(\frac{P_H(1)}{P_H(0)}\right)$$

Usamos la propiedad $-\ln(a)=\ln\left(\frac{1}{a}\right)$:

$$-\ln\left(\frac{P_H(1)}{P_H(0)}\right)=\ln\left(\frac{P_H(0)}{P_H(1)}\right)$$

Entonces:

$$y\ln\left(\frac{\lambda_1}{\lambda_0}\right) \ge (\lambda_1-\lambda_0) + \ln\left(\frac{P_H(0)}{P_H(1)}\right)$$

Como en este problema $\lambda_1>\lambda_0$, entonces $\frac{\lambda_1}{\lambda_0}>1$ y por eso $\ln\left(\frac{\lambda_1}{\lambda_0}\right)>0$. Al dividir por una cantidad positiva, la desigualdad no cambia de sentido:

$$y \ge \frac{(\lambda_1-\lambda_0) + \ln\left(\frac{P_H(0)}{P_H(1)}\right)}{\ln\left(\frac{\lambda_1}{\lambda_0}\right)}$$

Por lo tanto, el nivel de decision MAP teorico es:

$$\boxed{\gamma_{MAP}=\frac{(\lambda_1-\lambda_0) + \ln\left(\frac{P_H(0)}{P_H(1)}\right)}{\ln\left(\frac{\lambda_1}{\lambda_0}\right)}}$$


## 4. Reemplazo numerico del umbral MAP

Usamos:

$$\lambda_0=1, \qquad \lambda_1=3, \qquad P_H(0)=0.7, \qquad P_H(1)=0.3$$

Partimos de la formula:

$$\gamma_{MAP}=\frac{(\lambda_1-\lambda_0) + \ln\left(\frac{P_H(0)}{P_H(1)}\right)}{\ln\left(\frac{\lambda_1}{\lambda_0}\right)}$$

Reemplazamos cada valor:

$$\gamma_{MAP}=\frac{(3-1) + \ln\left(\frac{0.7}{0.3}\right)}{\ln\left(\frac{3}{1}\right)}$$

Resolvemos las restas y divisiones internas:

$$\gamma_{MAP}=\frac{2 + \ln\left(\frac{7}{3}\right)}{\ln(3)}$$

Calculamos los logaritmos:

$$\ln\left(\frac{7}{3}\right) \approx 0.8473$$

$$\ln(3) \approx 1.0986$$

Entonces:

$$\gamma_{MAP}\approx \frac{2 + 0.8473}{1.0986}$$

$$\gamma_{MAP}\approx \frac{2.8473}{1.0986}$$

$$\boxed{\gamma_{MAP}\approx 2.5917}$$

Como $Y$ es discreta y toma valores enteros, la regla practica equivalente es:

$$\hat H=1 \quad \text{si} \quad Y \ge 3$$

$$\hat H=0 \quad \text{si} \quad Y < 3$$


In [ ]:
gamma_map = ((lambda1 - lambda0) + np.log(p0 / p1)) / np.log(lambda1 / lambda0)
gamma_map_entero = ceil(gamma_map)

print(f"Nivel de decision MAP teorico: gamma_MAP = {gamma_map:.4f}")
print(f"Regla discreta equivalente: decidir H=1 si Y >= {gamma_map_entero}")

## 5. Analisis previo con histogramas

Antes de usar el generador de hipotesis con probabilidades distintas, conviene mirar los histogramas de las muestras bajo cada hipotesis.

Esto ayuda a interpretar por que el umbral MAP queda mas alto que el umbral ML: como $H=0$ es mas probable que $H=1$, el receptor exige una observacion mas grande para decidir $H=1$.


In [ ]:
samples_h0 = rng.poisson(lambda0, size=100_000)
samples_h1 = rng.poisson(lambda1, size=100_000)

plt.figure(figsize=(12, 4))
plt.hist(samples_h0, bins=np.arange(-0.5, 11.5, 1), density=True, alpha=0.55, label=r"$Y|H=0$, $\lambda_0=1$")
plt.hist(samples_h1, bins=np.arange(-0.5, 11.5, 1), density=True, alpha=0.55, label=r"$Y|H=1$, $\lambda_1=3$")
plt.axvline(gamma_map, color="black", linestyle="--", label=fr"$\gamma_{{MAP}}={gamma_map:.4f}$")
plt.axvline(gamma_map_entero - 0.5, color="#D55E00", linestyle=":", label=fr"frontera discreta: $Y\geq {gamma_map_entero}$")
plt.xticks(range(0, 12))
plt.xlabel("y")
plt.ylabel("Frecuencia relativa")
plt.title("Histogramas normalizados de las observaciones condicionadas")
plt.legend()
plt.show()

## 6. Simulacion Monte Carlo usando el nivel MAP

El sistema simulado tiene tres bloques:

1. Generador de hipotesis con $P_H(0)=0.7$ y $P_H(1)=0.3$.
2. Canal Poisson, donde la intensidad depende de la hipotesis transmitida.
3. Receptor MAP, que decide usando el umbral calculado.

Para generar hipotesis con probabilidades distintas se usa una variable uniforme $Z \sim U(0,1)$.

Si $Z \ge 0.7$, generamos $H=1$; si $Z<0.7$, generamos $H=0$.

Entonces:

$$P(H=0)=P(Z<0.7)=0.7$$

$$P(H=1)=P(Z\ge 0.7)=0.3$$


In [ ]:
z = rng.uniform(size=nb_samples)
hypothesis = np.array([1 if z[i] >= p0 else 0 for i in range(len(z))])

print("Primeras 30 hipotesis generadas:")
print(hypothesis[:30])

print(f"Frecuencia relativa de H=0: {np.mean(hypothesis == 0):.4f}")
print(f"Frecuencia relativa de H=1: {np.mean(hypothesis == 1):.4f}")

In [ ]:
y = np.where(
    hypothesis == 1,
    rng.poisson(lambda1, size=nb_samples),
    rng.poisson(lambda0, size=nb_samples),
)

hypothesis_decoded_map = np.array([1 if y[i] >= gamma_map else 0 for i in range(len(y))])
prob_error_map_mc = np.mean(hypothesis_decoded_map != hypothesis)

print("Primeras 30 observaciones Y:")
print(y[:30])

print("Primeras 30 hipotesis decididas con MAP:")
print(hypothesis_decoded_map[:30])

print(f"Probabilidad de error Monte Carlo con MAP: {prob_error_map_mc:.4f}")

## 7. Calculo teorico de la probabilidad de error MAP

La simulacion Monte Carlo estima la probabilidad de error contando errores. Pero tambien podemos calcular el valor teorico para confirmar que el resultado tiene sentido.

Con la regla discreta:

$$\hat H=1 \quad \text{si} \quad Y\ge 3$$

$$\hat H=0 \quad \text{si} \quad Y<3$$

Hay dos formas de equivocarse:

- Falsa alarma: se decide $1$ cuando se transmitio $0$.
- Error por omision: se decide $0$ cuando se transmitio $1$.

Entonces:

$$P_e=P_H(0)P(\hat H=1|H=0)+P_H(1)P(\hat H=0|H=1)$$

Reemplazando la regla de decision:

$$P_e=P_H(0)P(Y\ge 3|H=0)+P_H(1)P(Y<3|H=1)$$

Como $Y|H=0 \sim \text{Poisson}(1)$:

$$P(Y\ge 3|H=0)=1-P(Y<3|H=0)$$

$$P(Y\ge 3|H=0)=1-[P(Y=0|H=0)+P(Y=1|H=0)+P(Y=2|H=0)]$$

$$P(Y\ge 3|H=0)=1-\left[\frac{1^0}{0!}e^{-1}+\frac{1^1}{1!}e^{-1}+\frac{1^2}{2!}e^{-1}\right]$$

$$P(Y\ge 3|H=0)=1-e^{-1}\left[1+1+\frac{1}{2}\right]$$

$$P(Y\ge 3|H=0)=1-2.5e^{-1}$$

Como $Y|H=1 \sim \text{Poisson}(3)$:

$$P(Y<3|H=1)=P(Y=0|H=1)+P(Y=1|H=1)+P(Y=2|H=1)$$

$$P(Y<3|H=1)=\frac{3^0}{0!}e^{-3}+\frac{3^1}{1!}e^{-3}+\frac{3^2}{2!}e^{-3}$$

$$P(Y<3|H=1)=e^{-3}\left[1+3+\frac{9}{2}\right]$$

$$P(Y<3|H=1)=8.5e^{-3}$$

Por lo tanto:

$$P_e=0.7(1-2.5e^{-1})+0.3(8.5e^{-3})$$

$$\boxed{P_e \approx 0.183}$$


In [ ]:
def poisson_pmf(k, lam):
    return (lam ** k) * exp(-lam) / factorial(k)

def poisson_cdf_less_than(gamma_entero, lam):
    return sum(poisson_pmf(k, lam) for k in range(gamma_entero))

def theoretical_error(gamma_real, lambda0, lambda1, p0, p1):
    gamma_entero = ceil(gamma_real)
    p_false_alarm = 1 - poisson_cdf_less_than(gamma_entero, lambda0)
    p_miss = poisson_cdf_less_than(gamma_entero, lambda1)
    return p0 * p_false_alarm + p1 * p_miss, p_false_alarm, p_miss, gamma_entero

prob_error_map_teorica, p_fa_map, p_miss_map, gamma_map_entero = theoretical_error(gamma_map, lambda0, lambda1, p0, p1)

print(f"P_FA MAP = P(Y >= {gamma_map_entero}|H=0) = {p_fa_map:.4f}")
print(f"P_miss MAP = P(Y < {gamma_map_entero}|H=1) = {p_miss_map:.4f}")
print(f"Probabilidad de error teorica MAP = {prob_error_map_teorica:.4f}")

## 8. Comparacion cambiando solo el nivel de decision por el caso ML

La regla ML es un caso particular de MAP cuando las hipotesis son equiprobables:

$$P_H(0)=P_H(1)$$

Si $P_H(0)=P_H(1)$, entonces:

$$\ln\left(\frac{P_H(0)}{P_H(1)}\right)=\ln(1)=0$$

La formula MAP se reduce a:

$$\gamma_{ML}=\frac{\lambda_1-\lambda_0}{\ln\left(\frac{\lambda_1}{\lambda_0}\right)}$$

Reemplazando:

$$\gamma_{ML}=\frac{3-1}{\ln\left(\frac{3}{1}\right)}$$

$$\gamma_{ML}=\frac{2}{\ln(3)}$$

$$\gamma_{ML}\approx \frac{2}{1.0986}$$

$$\boxed{\gamma_{ML}\approx 1.8205}$$

Como $Y$ es discreta, usar $Y\ge \gamma_{ML}$ equivale a decidir $H=1$ si:

$$Y\ge 2$$

Importante: para que el caso sea realmente equiprobable debe cumplirse:

$$P_H(0)=P_H(1)=0.5$$

Por eso, en esta parte se calculan las hipotesis y la probabilidad de error usando $P_H(0)=0.5$ y $P_H(1)=0.5$.

Si se mantuvieran $P_H(0)=0.7$ y $P_H(1)=0.3$ y solamente se cambiara el umbral, eso no seria un problema equiprobable: seria usar el umbral ML dentro de un problema cuyas probabilidades reales siguen siendo no equiprobables.


In [ ]:
p0_ml = 0.5
p1_ml = 0.5

gamma_ml = ((lambda1 - lambda0) + np.log(p0_ml / p1_ml)) / np.log(lambda1 / lambda0)
gamma_ml_entero = ceil(gamma_ml)

z_ml = rng.uniform(size=nb_samples)
hypothesis_ml = np.array([1 if z_ml[i] >= p0_ml else 0 for i in range(len(z_ml))])

y_ml = np.where(
    hypothesis_ml == 1,
    rng.poisson(lambda1, size=nb_samples),
    rng.poisson(lambda0, size=nb_samples),
)

hypothesis_decoded_ml = np.array([1 if y_ml[i] >= gamma_ml else 0 for i in range(len(y_ml))])
prob_error_ml_mc = np.mean(hypothesis_decoded_ml != hypothesis_ml)

prob_error_ml_teorica, p_fa_ml, p_miss_ml, gamma_ml_entero = theoretical_error(gamma_ml, lambda0, lambda1, p0_ml, p1_ml)

print(f"Probabilidad usada para ML: P_H(0) = {p0_ml:.1f}, P_H(1) = {p1_ml:.1f}")
print(f"Nivel de decision ML teorico: gamma_ML = {gamma_ml:.4f}")
print(f"Regla discreta equivalente: decidir H=1 si Y >= {gamma_ml_entero}")
print(f"Probabilidad de error Monte Carlo usando umbral ML: {prob_error_ml_mc:.4f}")
print(f"Probabilidad de error teorica usando umbral ML: {prob_error_ml_teorica:.4f}")

In [ ]:
labels = ["MAP\nP0=0.7, P1=0.3", "ML\nP0=0.5, P1=0.5"]
mc_errors = [prob_error_map_mc, prob_error_ml_mc]
theoretical_errors = [prob_error_map_teorica, prob_error_ml_teorica]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(x - width/2, mc_errors, width, label="Monte Carlo")
plt.bar(x + width/2, theoretical_errors, width, label="Teorico")
plt.xticks(x, labels)
plt.ylabel("Probabilidad de error")
plt.title("Comparacion de probabilidad de error")
plt.ylim(0, max(theoretical_errors) * 1.25)
plt.legend()
plt.show()

print(f"Resumen MAP: gamma = {gamma_map:.4f}, P0 = {p0:.1f}, P1 = {p1:.1f}, Pe_MC = {prob_error_map_mc:.4f}, Pe_teorica = {prob_error_map_teorica:.4f}")
print(f"Resumen ML : gamma = {gamma_ml:.4f}, P0 = {p0_ml:.1f}, P1 = {p1_ml:.1f}, Pe_MC = {prob_error_ml_mc:.4f}, Pe_teorica = {prob_error_ml_teorica:.4f}")

## 9. Conclusion

Para el caso MAP con $\lambda_0=1$, $\lambda_1=3$, $P_H(0)=0.7$ y $P_H(1)=0.3$, el nivel de decision obtenido es:

$$\boxed{\gamma_{MAP}=2.5917}$$

Como la observacion $Y$ es discreta, en la simulacion esto equivale a decidir $H=1$ cuando $Y\ge 3$.

La probabilidad de error obtenida por Monte Carlo queda cercana al valor teorico:

$$\boxed{P_e \approx 0.183}$$

Para el caso ML, las hipotesis deben ser equiprobables:

$$P_H(0)=P_H(1)=0.5$$

Con esas probabilidades, el nivel de decision ML es:

$$\gamma_{ML}=1.8205$$

que en forma discreta equivale a decidir $H=1$ cuando $Y\ge 2$.

La comparacion importante es conceptual: MAP usa las probabilidades a priori y por eso, cuando $P_H(0)=0.7$ y $P_H(1)=0.3$, desplaza el umbral hacia $Y\ge 3$. ML corresponde al caso equiprobable y por eso usa el umbral $Y\ge 2$.
